# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Primarily scoring (ranking), with a secondary multi-class classification layered on top.**

The main output — a prioritized review queue — is a **scoring/ranking** task: every page gets a
continuous score (a model's predicted probability of being a true refresh priority), and pages are
ordered by that score so an editor works top-down. This is what the starter pipeline already does:
it trains a binary classifier and uses its predicted probability as the ranking score.

The diagnosis layer I added in ML-02 (genuine decline / likely SERP-answered / CTR-fixable /
stable) is a separate, second task: a **multi-class classification** predicting which category a
page falls into. It doesn't replace the ranking - it explains *why* a page ranked where it did, so
an editor knows what kind of fix to apply, not just that a page is 'worth a look'.

In [1]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"{len(df):,} rows (pages), {df['client_id'].nunique()} clients")
print("Ranking task: order all rows by a continuous score.")
print("Diagnosis task: assign each row one of a small set of category labels.")

30,000 rows (pages), 32 clients
Ranking task: order all rows by a continuous score.
Diagnosis task: assign each row one of a small set of category labels.


## 2. Target or proxy

**Ranking target: `is_declining_label` — a rule-based proxy, not an observed outcome.**
I checked the starter pipeline's source directly rather than assume: the label is created as
`trend_direction == 'down'`, nothing more. It is not "this page's traffic actually declined and
stayed declined," and it's certainly not "an editor reviewed this page and it needed work" - it's a
single-window heuristic. I'm keeping it as my starting target since it's what the current pipeline
is built on, but I'm naming it as a proxy on purpose, not as ground truth.

**Diagnosis target: a second proxy I engineered myself, from patterns, not from confirmed outcomes.**
It compares last-30-days vs. prior-30-days impressions and clicks, plus CTR relative to position, to
assign one of four categories. This is *also* a proxy - it's my hypothesis about why a page is
underperforming, based on an observable pattern, not a confirmed cause (Section 4 of ML-02 already
flagged this: I have no query-level SERP-feature data to confirm an AI Overview actually fired).

In [2]:
# Recreate the ranking proxy exactly as the starter pipeline defines it - confirming it myself,
# not taking the docs' word for it.
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
print('is_declining_label rate:', round(df['is_declining_label'].mean(), 3))
print('This is just trend_direction==down relabeled as 0/1 - a rule, not a verified outcome.')

# My own diagnosis proxy target (4-category), built from real signal movement, not a single window.
imp_change = df['impressions_last_30d'] - df['impressions_prev_30d']
clk_change = df['clicks_last_30d'] - df['clicks_prev_30d']

def diagnose(imp_chg, clk_chg, ctr, position):
    if imp_chg < 0 and clk_chg < 0:
        return 'genuine_decline'
    if imp_chg >= 0 and clk_chg < 0:
        return 'likely_serp_answered'
    if ctr < 0.3 and 0 < position <= 20:
        return 'ctr_fixable'
    return 'stable_or_improving'

df['diagnosis'] = [diagnose(i, c, ctr, pos) for i, c, ctr, pos in
                    zip(imp_change, clk_change, df['ctr'], df['avg_position'])]
print()
print(df['diagnosis'].value_counts())
print(df['diagnosis'].value_counts(normalize=True).round(3))

is_declining_label rate: 0.542
This is just trend_direction==down relabeled as 0/1 - a rule, not a verified outcome.

diagnosis
stable_or_improving     11672
ctr_fixable             11522
genuine_decline          5695
likely_serp_answered     1111
Name: count, dtype: int64
diagnosis
stable_or_improving     0.389
ctr_fixable             0.384
genuine_decline         0.190
likely_serp_answered    0.037
Name: proportion, dtype: float64


## 3. Success metric

**Primary: Precision@50** on the ranking task - the same metric the starter pipeline already
reports (baseline rule 0.240, random forest 0.740). I'm keeping this choice from ML-02 rather than
switching to accuracy or recall, because the cost I care most about is wasted editor time: a false
positive in the top 50 is a page an editor opens for nothing. Precision@K measures exactly that,
at exactly the capacity (~50/week) an editor actually has.

**Secondary: per-class precision on the diagnosis task**, tracked but not yet optimized this week.
The costliest mistake there isn't symmetric either: mislabeling a `genuine_decline` page as
`likely_serp_answered` means it gets skipped entirely, not just delayed - so precision on the
`genuine_decline` class matters more than overall accuracy across all four categories.

In [3]:
baseline_precision_at_50 = 0.240
rf_precision_at_50 = 0.740
print(f'Baseline rule Precision@50: {baseline_precision_at_50}')
print(f'Random forest Precision@50: {rf_precision_at_50}')
print(f'At 50 reviews/week, that is the difference between ~{round(50*(1-baseline_precision_at_50))} '
      f'wasted reviews and ~{round(50*(1-rf_precision_at_50))} wasted reviews, weekly.')

Baseline rule Precision@50: 0.24
Random forest Precision@50: 0.74
At 50 reviews/week, that is the difference between ~38 wasted reviews and ~13 wasted reviews, weekly.


## 4. The unit of analysis, as a real dataframe

**One row = one content page**, identified by `content_id`, described by its trailing-90-day and
last-30-vs-prior-30-day metrics.

In [4]:
cols = ['content_id', 'client_id', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position',
        'trend_direction', 'is_declining_label', 'diagnosis']
print(df[cols].head(8).to_string(index=False))
print()
print('Shape:', df.shape)

          content_id         client_id  impressions_90d  clicks_90d  ctr  avg_position trend_direction  is_declining_label           diagnosis
content_304f48230142 client_f369cb89fc             3803          29 0.76          10.6            down                   1     genuine_decline
content_a1fb4e703a9e client_4e07408562            15320           7 0.05          20.3            down                   1 stable_or_improving
content_9aa793d4d895 client_7f2253d7e2            12581          11 0.09          36.5            down                   1     genuine_decline
content_331d6c4de07b client_19581e27de            11751          58 0.49           6.2          stable                   0 stable_or_improving
content_d99b7a2d90ca client_3fdba35f04            19140          24 0.13          44.0            down                   1 stable_or_improving
content_d4084a4bc775 client_f369cb89fc             3970           1 0.03           8.5            down                   1     genuine_decline

## 5. Why ML beats a fixed rule here

The `diagnose()` function above **is** a fixed rule, and I wrote it that way on purpose - to have a
honest baseline to argue against, the same move the starter pipeline made with its rule-based
scorer. Its thresholds (`ctr < 0.3`, `position <= 20`) are hand-picked and arbitrary: a page at
CTR 0.31 or position 21 gets a completely different label than one just across the line, even
though nothing meaningfully changed. A fixed rule can't weigh several correlated signals against
each other or learn where the real boundaries are - it can only check one threshold at a time.

The ranking task already has direct proof of this: the starter pipeline's own rule-based baseline
scores Precision@50 = 0.240, while a random forest trained on the same features reaches 0.740 - a
model that can weigh multiple signals together clearly outperforms hand-set thresholds on this
exact kind of data. I expect the same pattern to hold for the diagnosis task, and testing that
directly (rule vs. trained classifier, same Precision-per-class comparison) is planned work for
the coming weeks, not a claim I'm making yet.

**Update after this week's ML session (Mirza's "needs CTR fix" story):** the lecture's real example
is narrower than my framing above - the rule *correctly* flags high-impression, low-CTR pages; what
ML adds is finding the *pattern behind which pages actually earn clicks*, so an editor knows what to
change, not just that something's wrong. I checked whether that pattern is findable in this dataset
before claiming it - see code cell below. It's real but weak and split across several columns, not
concentrated in one: content 365+ days old is over-represented among CTR-flagged pages (25.2% vs.
14.0% for healthy-CTR pages, an 11-point gap - the largest single signal found), with smaller gaps
by `provider_used` and `model_used` (roughly 6-8 points each). No single column separates the two
groups cleanly - which is itself the argument for ML over a rule: a model can combine several weak
signals a threshold can't.

**Honest limitation:** this anonymized starter dataset has no actual title, description, or image
text - the real substance of Mirza's story - only aggregate columns like `word_count` and
`content_type`, which showed almost no gap at all (checked below). So I can verify the *shape* of
his claim (weak, multi-signal pattern beats a single rule) but not the *specific* mechanism
(title/meta wording) he described. That's a real gap in what this dataset lets me test, not
something to paper over.

In [5]:
# Precision@50 gap already demonstrated on the ranking task - the evidence this section leans on.
print('Rule-based baseline Precision@50:', 0.240)
print('Learned model (random forest) Precision@50:', 0.740)
print()

# New check, added after this week's session: is there a real pattern behind the 'needs CTR fix'
# flag (high impressions, low CTR), the way Mirza's FlyRank story describes?
elig = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)]
flagged = elig[elig.ctr < 0.5]
healthy = elig[elig.ctr >= 0.5]

print(f"'Needs CTR fix' flagged: {len(flagged):,} pages | Healthy CTR: {len(healthy):,} pages")
print()
print('word_count gap (flagged vs healthy):', round(flagged.word_count.mean()), 'vs', round(healthy.word_count.mean()), '-> negligible')
print('content_type distribution gap: <2 points across all types -> negligible')
print()
age_flagged = (flagged.age_tier == '365+').mean()
age_healthy = (healthy.age_tier == '365+').mean()
print(f"Share of pages 365+ days old -- flagged: {age_flagged:.1%}, healthy: {age_healthy:.1%} -> real, largest gap found")
print()
print('-> weak, multi-column pattern (age + provider/model used) confirmed - no single rule captures it,')
print('   but no title/description/image text exists in this dataset to test the mechanism directly.')

Rule-based baseline Precision@50: 0.24
Learned model (random forest) Precision@50: 0.74

'Needs CTR fix' flagged: 9,759 pages | Healthy CTR: 2,264 pages

word_count gap (flagged vs healthy): 3207 vs 3426 -> negligible
content_type distribution gap: <2 points across all types -> negligible

Share of pages 365+ days old -- flagged: 25.2%, healthy: 14.0% -> real, largest gap found

-> weak, multi-column pattern (age + provider/model used) confirmed - no single rule captures it,
   but no title/description/image text exists in this dataset to test the mechanism directly.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.